# Does the premium's path carry more than its summaries?

[`06_linear`](06_linear.ipynb) fitted a design matrix that is mostly one economic quantity - the
**premium**, the gap between the perpetual price and spot that the funding payment is computed
from - measured many ways. Among those ways are hand-built summaries of the premium's recent
*path*: its change over six horizons, its volatility over four, its z-score over two windows, its
quantile position over three. Each of those columns compresses a stretch of history into one
number, and a human chose the compression.

A sequence model does not take that compression as given. It reads the last 60 settlements of
every feature as an ordered window and learns its own summary. So the question this notebook and
[`10_dl_tcn`](10_dl_tcn.ipynb) put to the data is narrow and answerable: **on this case study,
does a learned representation of the path beat the hand-built one?** Not "are neural networks
useful" - the design matrix already contains a considerable amount of path information, and the
sequence family has to earn its keep against that, not against a naive baseline.

Two architectures are fitted here against the same request:


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 溢價的變化路徑是否比其摘要包含更多資訊？

[`06_linear`](06_linear.ipynb) 擬合的設計矩陣，主要是以多種方式衡量同一經濟量：**溢價**，即永續合約價格與現貨的差距，也是計算資金費用所依據的數值。其中包括人為建立、用來概括溢價近期*變化路徑*的特徵：六種預測期間的變化、四種視窗的波動率、兩種視窗的 z 分數、三種分位數位置。每個欄位都將一段歷史資料壓縮成一個數字，壓縮方式由人選擇。

序列模型不將這種壓縮視為既定做法。它將每項特徵最近 60 次結算讀作有順序的視窗，並學習自己的摘要。因此，本 Notebook 與 [`10_dl_tcn`](10_dl_tcn.ipynb) 對資料提出的問題明確而可回答：**在此個案研究中，學得的路徑表示是否勝過人工建立的表示？** 問題不是「神經網路是否有用」；設計矩陣已包含大量路徑資訊，序列模型類別必須相對於此證明價值，而非只勝過天真的基準。

此處針對相同請求擬合兩種架構：
<!-- ML4T_ZH_END c0000-p000 -->

- **NLinear** is the baseline, and it is deliberately almost nothing. It subtracts the last value
  of each window from the window, applies a single linear map to what remains, and adds the
  subtracted value back. It has no recurrence, no gating and no nonlinearity. It exists so that
  "the LSTM did better" has to mean better than the simplest thing that reads the same window in
  the same order - which, on financial series, is a bar a great many published architectures do
  not clear.
- **LSTM** is the recurrent model: two layers, a hidden state of 64, dropout 0.1. It processes
  the window one settlement at a time and carries a state forward, so unlike NLinear it can in
  principle represent an interaction between what happened early in the window and what happened
  late.

Both go through the same request contract - the same feature order, the same folds, the same
missing-observation policy, the same checkpoint schedule. **That is the point of running them
from one notebook.** When the two differ in a later backtest, the difference is the architecture,
because nothing else was allowed to vary.

## The grid is 8-hourly, and gaps in it are real

A perpetual's funding is settled every 8 hours, and this case study's observation grid is that
settlement cadence. A lookback of 60 is therefore **60 settlements, about 20 days** - not 60 days
and not 60 rows of whatever happened to be adjacent in the file.


<!-- ML4T_ZH_BEGIN c0000-p001 -->
- **NLinear** 是基準，刻意設計得極為簡單。它從視窗各數值減去最後一個值，對剩餘數值套用一次線性映射，再把先前減去的值加回。它沒有遞迴、門控或非線性。設立它是為了讓「LSTM 表現較好」必須表示勝過以相同順序讀取相同視窗的最簡單方法；在金融序列上，許多已發表架構都未跨過這個門檻。
- **LSTM** 是遞迴模型：兩層、隱藏狀態維度 64、dropout 為 0.1。它逐次結算處理視窗，將狀態向前傳遞，因此理論上可表示視窗前段與後段事件之間的交互作用，這是 NLinear 做不到的。

兩者遵循相同請求契約：特徵順序、切分、缺漏觀測值政策及檢查點時程均相同。**這就是在同一份 Notebook 執行兩者的理由。** 若兩者在後續回測中出現差異，差異來自架構，因為其他條件都不允許變動。

## 時間網格每八小時一格，缺口是真實存在的

永續合約資金費用每八小時結算一次，本個案研究的觀測時間網格依此結算節奏建立。因此回溯長度 60 指**60 次結算，約 20 天**，不是 60 天，也不是檔案中恰好相鄰的 60 列。
<!-- ML4T_ZH_END c0000-p001 -->

That distinction has teeth here. A perpetual can be delisted, halted, or newly listed, and the
exchange's history has holes. If a 60-bar window were built by taking 60 adjacent *rows*, a
window spanning a two-day outage would silently splice across it and present the model with a
discontinuity as though it were a normal step. The resolved policy on every request below is
`exclude_windows_crossing_missing_expected_periods`: a window that would cross a settlement the
grid expects and the data does not have is **dropped, not imputed**. The eligible-row count in
the contracts table is what survives that rule, and it is smaller than the row count of the
panel.

## A checkpoint is a model, not a progress marker

Each configuration trains for 100 epochs and persists its state every 5, so each produces 20
checkpoints, and **each checkpoint is a distinct prediction identity** that a later backtest can
select. Early stopping is not implemented as a rule that halts training; it is implemented as a
population of checkpoints from which selection picks. That is why the population is frozen before
the first fit: a checkpoint that trains and then turns out to be poor stays in the population it
was declared in, and cannot quietly disappear from the count it is judged against.

**Learning objectives.** By the end of this notebook you will be able to:


<!-- ML4T_ZH_BEGIN c0000-p002 -->
此處差異有實質影響。永續合約可能下架、暫停交易或剛上架，交易所歷史資料也有缺口。若只取相鄰的 60 *列*建立視窗，橫跨兩天資料中斷的視窗會默默拼接兩端，將不連續變動當成一般步驟呈給模型。下方每項請求解析後的政策是 `exclude_windows_crossing_missing_expected_periods`：若視窗會跨過時間網格預期應有、資料卻缺少的結算時點，就**捨棄該視窗，而非插補**。合約表中的合格資料列數，就是套用此規則後留下的數量，少於面板資料列總數。

## 檢查點是模型，而非進度標記

每種設定訓練 100 個週期，每 5 個週期持久化一次狀態，因此各產生 20 個檢查點，且**每個檢查點都是可供後續回測選擇的獨立預測識別值**。提前停止並非實際中斷訓練的規則，而是從檢查點群組中進行選擇。因此模型群組在首次擬合前就先固定：某個檢查點訓練完成後，即使表現不佳，仍保留在原本宣告的群組中，不能默默從評估時所計入的候選數量消失。

**學習目標。** 完成本 Notebook 後，你將能夠：
<!-- ML4T_ZH_END c0000-p002 -->

- Explain why a sequence model on an irregular observation grid needs a declared cadence, and
  what goes wrong when window construction uses row adjacency instead.
- Read a resolved sequence request and say what lookback, gap policy and eligible row count the
  run will actually use, before anything is fitted.
- Say what a checkpoint schedule buys, and why every checkpoint is registered as its own
  prediction set rather than only the last or the best.
- Recognise that a linear baseline sharing the sequence contract is the correct comparison for a
  recurrent model, and that beating a cross-sectional model is not the same claim.

**Book reference:** Chapter 19, recurrent neural networks for time series.

**Prerequisites:** [`03_financial_features`](03_financial_features.ipynb) and
[`04_model_based_features`](04_model_based_features.ipynb) have written the feature matrices, and
[`05_evaluation`](05_evaluation.ipynb) has established the walk-forward folds. The canonical run
uses CUDA; the reduced run in CI does not.

**What it writes:** one training run per configuration and one complete validation prediction set
per checkpoint, in `run_log/registry.db` and under `run_log/training/` and
`run_log/predictions/`, grouped under a named population.
[`13_backtest`](13_backtest.ipynb) reads that population and selects on validation backtest
Sharpe. **Selection happens there, not here.** Nothing in this notebook ranks anything.

<!-- ML4T_ZH_BEGIN c0000-p003 -->
- 解釋為何在不規則觀測網格上使用序列模型，必須宣告觀測節奏；若改以相鄰資料列建立視窗，會發生什麼問題。
- 讀取解析後的序列請求，在擬合前說明實際使用的回溯長度、缺口政策及合格資料列數。
- 說明檢查點時程的用途，以及為何每個檢查點都登錄為獨立預測集，而非只登錄最後或最佳者。
- 辨識採用相同序列契約的線性基準，是比較遞迴模型的適當對象；勝過橫斷面模型則是另一種主張。

**書籍參考：**第 19 章，時間序列的遞迴神經網路。

**先備知識：**[`03_financial_features`](03_financial_features.ipynb) 與 [`04_model_based_features`](04_model_based_features.ipynb) 已寫入特徵矩陣，[`05_evaluation`](05_evaluation.ipynb) 已確立滾動向前切分。標準執行使用 CUDA，CI 中縮小規模的執行則不用。

**寫入內容：**每種設定各有一次訓練執行，每個檢查點各有一套完整驗證預測結果，記錄在 `run_log/registry.db`，並存於 `run_log/training/` 與 `run_log/predictions/` 之下，依具名模型群組整理。[`13_backtest`](13_backtest.ipynb) 讀取該群組，依驗證回測夏普比率選擇。**選擇在該處發生，而非此處。** 本 Notebook 不對任何結果排名。
<!-- ML4T_ZH_END c0000-p003 -->


In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    REGRESSION_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    plan_specs,
    run_model_plan,
)
from case_studies.research import population_supersedes

In [2]:
EXECUTION_TIER = "canonical"
SUPERSEDES_POPULATION: str = "1b444ce334d4"
# The generation of this notebook's own checkpoint population that this run replaces, if any.
# Distinct from SUPERSEDES_POPULATION above, which is the case-wide official model population:
# the two are separate declarations and a refit can move either without moving the other.
SUPERSEDES_MODEL_POPULATION: str = "f3168d150e27"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = REGRESSION_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {"device": "cuda"}

## 1. Resolve the sequence and checkpoint identities

Nothing is fitted in this cell. `model_request_catalog` reads the configurations this case study
declares for the regression labels and returns the requests they resolve to; the plan that
follows binds those requests to the data on disk and computes an identity for each. Reading the
resolved plan before training is what makes the run auditable: if the lookback, the gap policy or
the eligible row count is not what you expected, you find out here rather than after the fits.

`config_prefix=("nlinear", "lstm")` is what restricts this notebook to the two architectures
discussed above. The TCN declared alongside them in `config/training/fwd_ret_8h.yaml` is fitted
by [`10_dl_tcn`](10_dl_tcn.ipynb) against the same contract.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 1. 解析序列與檢查點識別值

此儲存格不擬合任何模型。`model_request_catalog` 讀取本個案研究為迴歸標籤宣告的設定，傳回解析後的請求；後續計畫將請求連結到磁碟資料，並計算各自的識別值。在訓練前讀取解析後的計畫，讓執行可供稽核：若回溯長度、缺口政策或合格資料列數不符預期，會在此處發現，而非擬合完成後。

`config_prefix=("nlinear", "lstm")` 將本 Notebook 限制為上文討論的兩種架構。同時在 `config/training/fwd_ret_8h.yaml` 宣告的 TCN，則由 [`10_dl_tcn`](10_dl_tcn.ipynb) 依相同契約擬合。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(
        study,
        supersedes=population_supersedes(
            study,
            name="crypto-validation-predictions-v1",
            declared=SUPERSEDES_POPULATION,
        ),
    )
    if EXECUTION_TIER == "canonical"
    else None
)
requests = model_request_catalog("deep_learning", labels=LABELS, config_prefix=("nlinear", "lstm"))
requests

family,label,config_name
str,str,str
"""deep_learning""","""fwd_ret_8h""","""nlinear"""
"""deep_learning""","""fwd_ret_8h""","""lstm_h64"""
"""deep_learning""","""fwd_ret_24h""","""nlinear"""
"""deep_learning""","""fwd_ret_24h""","""lstm_h64"""


The table below is the run's declaration of what it is about to do. `gap_policy` and `lookback`
are read back out of the frozen specification rather than restated from the configuration file,
so the table cannot drift from what the fit will use. `eligible_rows` is the count of window
end-points that survive the gap rule - the effective sample the model is fitted on, which is
always smaller than the panel and is the number to quote when describing how much data a
sequence model here actually saw.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
下表是這次執行即將進行內容的宣告。`gap_policy` 與 `lookback` 直接從已固定規格讀回，而非從設定檔重新敘述，因此表格不會與實際擬合使用的設定分歧。`eligible_rows` 是通過缺口規則的視窗終點數量，亦即模型實際用來擬合的有效樣本；它一定小於面板資料列數，也是在說明此處序列模型實際看過多少資料時，應引用的數字。
<!-- ML4T_ZH_END c0005-p000 -->


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# Sequence eligibility follows from the resolved gap policy and lookback, so read both from the
# frozen specification instead of restating the configuration file here.
resolved_preprocessing = [spec["computation"]["preprocessing"] for spec in plan_specs(plan)]
contracts = declared_contracts(plan).with_columns(
    pl.Series("gap_policy", [step["gap_policy"] for step in resolved_preprocessing]),
    pl.Series("lookback", [step["lookback"] for step in resolved_preprocessing]),
)
contracts.select(
    "label",
    "config_name",
    "gap_policy",
    "lookback",
    "checkpoint_value",
    "eligible_rows",
    "training_hash",
)

label,config_name,gap_policy,lookback,checkpoint_value,eligible_rows,training_hash
str,str,str,i64,i64,i64,str
"""fwd_ret_8h""","""nlinear""","""calendar_grid_observation_mask…",60,5,32320,"""1314218b099a"""
"""fwd_ret_8h""","""nlinear""","""calendar_grid_observation_mask…",60,10,32320,"""1314218b099a"""
"""fwd_ret_8h""","""nlinear""","""calendar_grid_observation_mask…",60,15,32320,"""1314218b099a"""
"""fwd_ret_8h""","""nlinear""","""calendar_grid_observation_mask…",60,20,32320,"""1314218b099a"""
"""fwd_ret_8h""","""nlinear""","""calendar_grid_observation_mask…",60,25,32320,"""1314218b099a"""
…,…,…,…,…,…,…
"""fwd_ret_24h""","""lstm_h64""","""calendar_grid_observation_mask…",60,80,32266,"""748cd563b047"""
"""fwd_ret_24h""","""lstm_h64""","""calendar_grid_observation_mask…",60,85,32266,"""748cd563b047"""
"""fwd_ret_24h""","""lstm_h64""","""calendar_grid_observation_mask…",60,90,32266,"""748cd563b047"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
完整的個案研究模型群組在首次擬合前就已記錄，因此後續訓練失敗的成員，不能默默從已宣告群組中消失。本 Notebook 產生其中一部分，而且必須包含在完整宣告中。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## 2. Execute the declared population

The adapter fits each configuration on each fold, writes a checkpoint every fifth epoch, and
registers one complete validation prediction set per checkpoint. A fitted state is persisted with
a digest, and a cached state is reused only when the digest matches, so a resumed run cannot
quietly continue from a state that a code change has invalidated.

The completeness check below is the one that matters. A prediction set is `complete` when it
covers every eligible validation key for its fold; a set that covers most of them is not a
slightly worse result, it is a different sample, and comparing it against a full one would be
comparing two things measured on different data. The run raises rather than publishing a
population containing one.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
## 2. 執行已宣告的模型群組

轉接器在每個切分上擬合各設定，每五個訓練週期寫入一次檢查點，並為每個檢查點登錄一套完整驗證預測結果。擬合狀態連同摘要一起持久化，只有已快取狀態的摘要相符時才重用，因此接續執行不會默默沿用因程式碼變更而失效的狀態。

下方最重要的是完整性檢查。預測集涵蓋該切分的每個合格驗證鍵值時，才算 `complete`；只涵蓋大部分鍵值的結果並非稍差，而是使用不同樣本，若與完整結果比較，就等於比較在不同資料上衡量的兩個結果。執行過程會報錯，而非發布含有這種預測集的模型群組。
<!-- ML4T_ZH_END c0009-p000 -->


In [6]:
execution = run_model_plan(
    plan,
    supersedes=population_supersedes(
        study,
        name="crypto-lstm-validation-predictions-v1",
        declared=SUPERSEDES_MODEL_POPULATION,
    ),
    population_name="crypto-lstm-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("sequence baseline and LSTM checkpoint population is incomplete")
catalog.select(
    "label",
    "config_name",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

Fold-major CV: 2 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=21,457 seq across 16 symbols
    val=15,323 seq across 18 symbols
    creating datasets...
    datasets ready
    nlinear:


      epoch   1/100: train_loss=0.137056


      epoch   2/100: train_loss=0.088834


      epoch   3/100: train_loss=0.062069


      epoch   4/100: train_loss=0.044717


... (861 outputs left out)

      best_ep=5, IC=+0.0066 (113.0s, 20 checkpoints)


  lstm_h64: best_epoch=60, IC=+0.0128 (213.8s)



  Best: lstm_h64 @ epoch 60 (IC=+0.0128)
  Saved to case_studies/crypto_perps_funding/run_log/training/748cd563b047/diagnostics


label,config_name,checkpoint_value,training_hash,prediction_hash,complete
str,str,i64,str,str,bool
"""fwd_ret_24h""","""lstm_h64""",5,"""748cd563b047""","""8e3df100fa5f""",true
"""fwd_ret_24h""","""lstm_h64""",10,"""748cd563b047""","""bb61022ce8d6""",true
"""fwd_ret_24h""","""lstm_h64""",15,"""748cd563b047""","""ba74d9c5e069""",true
"""fwd_ret_24h""","""lstm_h64""",20,"""748cd563b047""","""99f5f5e2abd4""",true
"""fwd_ret_24h""","""lstm_h64""",25,"""748cd563b047""","""1e4e3045df05""",true
…,…,…,…,…,…
"""fwd_ret_8h""","""nlinear""",80,"""1314218b099a""","""793688c67b92""",true
"""fwd_ret_8h""","""nlinear""",85,"""1314218b099a""","""c30f300770a6""",true
"""fwd_ret_8h""","""nlinear""",90,"""1314218b099a""","""9f96779aeacc""",true


## Key takeaways and limitations

- **Eligibility follows the declared cadence, not row adjacency.** A 60-bar window is 60 expected
  8-hour settlements. A window that would cross a settlement missing from the data is dropped,
  which is why `eligible_rows` is smaller than the panel and why that count, not the panel
  height, is the sample size to quote.
- **The linear baseline is the comparison that means something.** NLinear reads the same window,
  in the same order, under the same contract, and has no recurrence at all. An LSTM that does not
  beat it has not shown that recurrence bought anything on this data.
- **Every checkpoint is a model.** Twenty per configuration, each registered as its own
  prediction identity, and selection among them happens in [`13_backtest`](13_backtest.ipynb) on
  validation backtest Sharpe. Reporting the best checkpoint's score as though one model had
  achieved it would be reporting a maximum over twenty draws as a single measurement.
- **The history is short and the folds are few.** This case study's usable perpetual funding
  history supports two validation folds, and a two-layer LSTM with a 64-unit hidden state has far
  more capacity than two folds of an 8-hourly panel can identify. Dropout and the checkpoint
  population are doing the regularization that a longer history would not need as badly.
- **A fixed lookback is a modelling assumption, not a neutral default.** Sixty settlements is
  about twenty days. Any dependence on something that happened before that window is invisible to
  these models by construction, however long the funding cycle they are meant to capture.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 重點摘要與限制

- **合格與否由預先指定的節奏決定，而非資料列相鄰與否。** 60 根 K 線視窗代表預期的 60 次八小時結算。若視窗會跨過資料中缺少的結算時點，就會被捨棄；因此 `eligible_rows` 小於面板資料列數，也因此應以該數量，而非面板高度，作為樣本數。
- **線性基準才是有意義的比較對象。** NLinear 在相同契約下，以相同順序讀取相同視窗，卻完全沒有遞迴。若 LSTM 未勝過它，就無法證明此資料因遞迴而受益。
- **每個檢查點都是一個模型。** 每種設定有 20 個，各有專屬預測識別值；[`13_backtest`](13_backtest.ipynb) 依驗證回測夏普比率選擇。若把最佳檢查點的分數當成單一模型成果，就等於把 20 次測量的最大值當成一次結果。
- **歷史資料短、切分少。** 本個案研究可用的永續合約資金費率歷史資料只支撐兩個驗證切分，而有兩層及 64 維隱藏狀態的 LSTM，其容量遠高於兩個切分的八小時面板所能識別。dropout 與檢查點群組提供正則化，彌補歷史資料不足所帶來的影響。
- **固定回溯長度是建模假設，不是中性的預設值。** 60 次結算約 20 天。依設計，這些模型看不到視窗之前發生事件的任何影響，不論要捕捉的資金費用週期有多長。
<!-- ML4T_ZH_END c0011-p000 -->
